In [1]:
import pandas as pd
import psycopg2
import pandas as pd
import geopandas as gpd
import requests
from geopandas.tools import sjoin
import time
import datetime
from collections import Counter 
import re
import random
import numpy as np
import unidecode
import matplotlib.pyplot as plt
import plotly.graph_objects as go 
import plotly.express as px

In [2]:
df_patients = pd.read_csv("../../data/data_octobre_2023/Pseudonymisation_provisoire.csv", sep=";",encoding_errors='ignore')

df_patients.rename(columns={'adresse':'ADRESS'}, inplace=True)

df_patients.dropna(subset="ADRESS")

,pseudo_provisoire,ADRESS,codepost,nom_commune_postal
0,1,34 RUE DES FRERES CHAUSSONS,92600,ASNIERES-SUR-SEINE
1,2,11 RUE EMILE DUBOIS,75014,PARIS
2,3,48 CHEMIN VERT,78680,EPONE
3,4,18 ALLEE DE LA CHARNILLE,47140,SAINT-SYLVESTRE-SUR-LOT
4,5,31 RUE DU GENERAL DE MIRIBEL,92500,RUEIL-MALMAISON
...,...,...,...,...
64289,64290,3 AV DE FOUILLEUSE,92210,SAINT-CLOUD
64290,64291,81 COTE DU TORCHON,27220,L'HABIT
64291,64292,60 RUE BAUDRICOURT,75013,Paris 13
64292,64293,159 AVENUE DE LA REPUBLIQUE,92320,CHATILLON


## Creates attributes lists

In [3]:
voirie = ["RUE","COURS","COUR","VOIE","RUELLE","ESPLANADE",
          "PLACE","SQUARE","SQ","ROND POINT","PL",
          "IMPASSE", "ALLEE", "CHEMIN" ,"ROUTE","RTE","IMP","PROMENADE","ALLE","ALL",
          "AVENUE", "BOULEVARD","BD","AVE","BLD","BVD","BLV","AVN","BV",
          "FERME","DOMAINE","LIEU DIT","QUARTIER","QUR"]

##ATT AVEC COUR : A LA FOIS DANS LES ADRESSES ET COMME VOIRIE   

numeros = ["1","2","3","4","5","6","7","8","9","0"]

bruit = [ "RESIDENCE","CHEZ","HOPITAL","SDF","RES","MME","BAT","MAISON","MR","HOTEL",
         "LOTISSEMENT","CENTRE","QUARTIER","APPT","APT","SANTE", "RETRAITE", "HOP","TRANSFERT"]

In [4]:
df = df_patients 
def find_attribute(attributes, text, is_number=False):
    if pd.notna(text):
        if is_number:
            numbers = re.findall(r'\d+', text)
            
            return ','.join(numbers) if numbers else ""
        else:
            found_attributes = [x for x in attributes if re.search(r'\b' + re.escape(x) + r'\b', text)]

            return ','.join(found_attributes) if found_attributes else ""
        
        return ""
    

    
df['voirie'] = df.apply(lambda x: find_attribute(voirie, x['ADRESS']), axis=1)
df['numeros'] = df.apply(lambda x: find_attribute(numeros, x['ADRESS'],is_number=True), axis=1)
df['bruit'] = df.apply(lambda x: find_attribute(bruit, x['ADRESS']), axis=1)

In [5]:
df_test = df[:10]
def find_pos_attributes(attributes, text,is_number=False):
    found_attributes = []
    
    if pd.notna(text):
        word = text.split()
        
        for i, word in enumerate(word):

            if is_number:
                if re.match(r'\d+', word):
                    found_attributes.append(i+1)

                #return ','.join(numbers) if numbers else ""
            elif word in attributes:
                 
                #found_attributes = [x for x in attributes if re.search(r'\b' + re.escape(x) + r'\b', text)]
                found_attributes.append(i+1)
                #return ','.join(found_attributes) if found_attributes else ""
                
        return found_attributes
        
df['pos_voirie'] = df.apply(lambda x: find_pos_attributes(voirie, x['ADRESS']), axis=1)
df['pos_numeros'] = df.apply(lambda x: find_pos_attributes(numeros, x['ADRESS'],is_number=True), axis=1)
df['pos_bruit'] = df.apply(lambda x: find_pos_attributes(bruit, x['ADRESS']), axis=1)
            

In [6]:
# def attribute_strpos(attributes, text, label):
#     if pd.notna(text):
#         attribute = next((x for x in attributes if x in text), "")
#         start = text.find(attribute)
#         end = start + len(attribute)

#         if attribute:
#             return [start, end, label]
#         else:
#             return


def attribute_strpos(attributes, text, label):
    if pd.notna(text):
        found_attributes = []
        for attribute in attributes:
            if attribute in text:
                start = text.find(attribute)
                end = start + len(attribute)
                found_attributes.append([start, end, label])
        return found_attributes
    else:
        return []

    
def annotate(text, attributes):
    tags = []

    for label, values in attributes.items():

        attribute = attribute_strpos(values, text, label)
        if attribute:
            tags.append(attribute)

    return tags


In [7]:
attributes = {
    'voirie': voirie,
    'numeros': numeros,
    'bruit': bruit,
}
df.dropna(subset="ADRESS")
df['label'] = df.apply(lambda x: annotate(x['ADRESS'], attributes), axis=1)

In [8]:
df

,pseudo_provisoire,ADRESS,codepost,nom_commune_postal,voirie,numeros,bruit,pos_voirie,pos_numeros,pos_bruit,label
0,1,34 RUE DES FRERES CHAUSSONS,92600,ASNIERES-SUR-SEINE,RUE,34,,[2],[1],[],"[[[3, 6, voirie]], [[0, 1, numeros], [1, 2, nu..."
1,2,11 RUE EMILE DUBOIS,75014,PARIS,RUE,11,,[2],[1],[],"[[[3, 6, voirie]], [[0, 1, numeros]]]"
2,3,48 CHEMIN VERT,78680,EPONE,CHEMIN,48,,[2],[1],[],"[[[3, 9, voirie]], [[0, 1, numeros], [1, 2, nu..."
3,4,18 ALLEE DE LA CHARNILLE,47140,SAINT-SYLVESTRE-SUR-LOT,ALLEE,18,,[2],[1],[],"[[[3, 8, voirie], [3, 7, voirie], [3, 6, voiri..."
4,5,31 RUE DU GENERAL DE MIRIBEL,92500,RUEIL-MALMAISON,RUE,31,,[2],[1],[],"[[[3, 6, voirie]], [[1, 2, numeros], [0, 1, nu..."
...,...,...,...,...,...,...,...,...,...,...,...
64289,64290,3 AV DE FOUILLEUSE,92210,SAINT-CLOUD,,3,,[],[1],[],"[[[0, 1, numeros]]]"
64290,64291,81 COTE DU TORCHON,27220,L'HABIT,,81,,[],[1],[],"[[[1, 2, numeros], [0, 1, numeros]]]"
64291,64292,60 RUE BAUDRICOURT,75013,Paris 13,RUE,60,,[2],[1],[],"[[[3, 6, voirie], [13, 17, voirie]], [[0, 1, n..."
64292,64293,159 AVENUE DE LA REPUBLIQUE,92320,CHATILLON,AVENUE,159,,[2],[1],[],"[[[4, 10, voirie], [4, 7, voirie]], [[0, 1, nu..."


In [9]:
# result = df.query('bruit.str.split().count(",") == 0')


#result = df[df["bruit"].str.split(',').str.len()==1]

result = df[df["bruit"].apply(lambda x: isinstance(x,str) and len(x.split(',')) >=5 and x.strip() != '')]
                              
                              # len(str(x).split(",") == 1 and pd.notna(x) and x.strip() != ''))]
len(result)

0

## Visualisation des positions des biais et leurs fréquences : 

In [10]:
def extract_position(row):
    position = {
        'voirie': [],
        'numeros': [],
        'bruit':[]
    }
    
    for all_label in row : 
        for label in all_label:          
            start, end, label_type = label
            position[label_type].append(start)
        
        
    return position

#df_test = df[:10]
positions_data = df['label'].apply(extract_position)

In [11]:
# Rassembler les données pour chaque type de bruit
voirie_positions = [pos for row in positions_data for pos in row['voirie']]
numeros_positions = [pos for row in positions_data for pos in row['numeros']]
bruit_positions = [pos for row in positions_data for pos in row['bruit']]

pd.DataFrame(voirie_positions).to_csv("./biais/pos_voirie.csv",sep=";")
pd.DataFrame(numeros_positions).to_csv("./biais/pos_numeros.csv",sep=";")
pd.DataFrame(bruit_positions).to_csv("./biais/pos_bruit.csv",sep=";")

# data_to_plot = [voirie_positions, numeros_positions,bruit_positions]

# plt.boxplot(data_to_plot, labels=['Voirie', 'Numeros','Bruit'])

# plt.title('Répartition des Positions des Bruits dans les Adresses')
# plt.ylabel('Position dans l\'Adresse')
# plt.show()

In [13]:
fig = go.Figure()
fig.add_trace(go.Box(y=voirie_positions, name="Voirie"))
fig.add_trace(go.Box(y=numeros_positions, name="Numéros"))
fig.add_trace(go.Box(y=bruit_positions, name="Bruit"))

fig.update_layout(
    title = "Répartition des positions du bruit, de la voirie et des numéros dans les adresses patients",
    yaxis_title = "Position dans l'adresse",
    boxmode ="group")

fig.write_image("images/box_plot_all.png")

In [20]:
fig = px.violin(bruit_positions, box=True)
fig.update_layout(
    title = "Répartition des positions du bruit dans les adresses patients",
    yaxis_title = "Position dans l'adresse",
    xaxis_title = "Bruit")
fig.write_image("images/violin_bruit.png")

In [21]:
fig = px.violin(voirie_positions, box=True)
fig.update_layout(
    title = "Répartition des positions des voiries dans les adresses patients",
    yaxis_title = "Position dans l'adresse",
    xaxis_title = "Voiries")
fig.write_image("images/violin_voirie.png")

In [22]:
fig = px.violin(numeros_positions, box=True)
fig.update_layout(
    title = "Répartition des positions des numéros dans les adresses patients",
    yaxis_title = "Position dans l'adresse",
    xaxis_title = "Numéros")
fig.write_image("images/violin_numeros.png")

In [17]:
#fig = go.Figure()
#fig.add_trace(px.violin(numeros_positions, box=True))
#fig.add_trace(px.violin(voirie_positions, box=True))
#fig.add_trace(px.violin(bruit_positions, box=True))

trace1 = go.Violin(y=numeros_positions, name="Numéros",box_visible=True)#,meanline=True)
trace2 = go.Violin(y=voirie_positions, name="Voirie",box_visible=True)#,meanline=True)
trace3 = go.Violin(y=bruit_positions, name="Bruit",box_visible=True)#,meanline=True)

fig = go.Figure(data = [trace1,trace2,trace3])

fig.update_layout(
    title = "Répartition des positions du bruit, de la voirie et des numéros dans les adresses patients",
    yaxis_title = "Position dans l'adresse",
)#boxmode ="group")

fig.write_image("images/violin_all.png")

## preparation des données pour doccano

In [ ]:
df.rename(columns={'ADRESS': 'text'}, inplace=True)
df['meta'] = ''
df['annotation_approver'] = ''
df[['text','meta','annotation_approver','label']].head()
# df.drop("voirie",axis=1)
# df.drop("numeros",axis=1)
#df.drop("bruit",axis=1)


In [ ]:
df_annotated = df 

In [ ]:
df_annotated.to_json('./NPL_adresses_annotated_all.json', orient='records', lines=True)